

JOMO KENYATTA UNIVERSITY OF
AGRICULTURE & TECHNOLOGY

CAT 2 - APRIL 2026

SEMESTER II, YEAR IV

BSC. DATA SCIENCE & ANALYTICS SDS 2412 : ANALYSIS OF LARGE DATASETS


MIKE MASYUKI SAMMY
SCT213-C002-002/2022






### QUESTION ONE (a): Construct DAG execution model and determine critical path across processing stages.

To construct a Directed Acyclic Graph (DAG) execution model for the given Distributed Systems Performance Dataset, we need to infer the dependencies between the tasks. The dataset provides `Node`, `Task Type`, `Execution Time (ms)`, `Memory (MB)`, `Throughput (ops/s)`, `Latency (ms)`, and `Replicas`. However, it doesn't explicitly state task dependencies. In a typical distributed system, some common dependencies can be assumed:

*   **ETL (Extract, Transform, Load)** often precedes data processing or analytics.
*   **Streaming Ingestion** feeds data for real-time processing.
*   **Batch Load** prepares data for batch processing or analytics.
*   **Analytics Query** and **Graph Query** typically operate on loaded/processed data.
*   **NoSQL Write (Cassandra)** could be a result of processing or a source for other queries.

Given the dataset, a simplified, plausible DAG could be:

1.  **Ingestion/Loading Stages:**
    *   `N1 (ETL)`
    *   `N2 (Streaming Ingestion)`
    *   `N3 (Batch Load)`

2.  **Processing/Storage Stages (depending on ingestion/load):**
    *   `N5 (NoSQL Write - Cassandra)` could follow `N1` or `N2` or `N3` if data is being prepared for Cassandra.

3.  **Query/Analytics Stages (depend on processed/stored data):**
    *   `N4 (Analytics Query)` could follow `N1`, `N3`, or `N5`.
    *   `N6 (Graph Query - Neo4j)` could follow `N1` or `N3` if graph data is prepared, or it could operate on data from `N5` if Cassandra is a data source for graph construction.

**Assumed Dependencies for DAG Construction:**
Let's make some reasonable assumptions to build a coherent DAG:
*   `N1 (ETL)` and `N3 (Batch Load)` are initial data preparation steps.
*   `N2 (Streaming Ingestion)` is an independent, continuous ingestion. For this model, we'll consider it leading to an independent processing path or feeding into N4/N5/N6 eventually.
*   `N5 (NoSQL Write - Cassandra)` depends on `N1 (ETL)` or `N3 (Batch Load)`.
*   `N4 (Analytics Query)` depends on `N1`, `N3`, or `N5`.
*   `N6 (Graph Query - Neo4j)` depends on `N1` or `N3` for data preparation.

**Simplified DAG Model:**

*   `Start` -> `N1 (ETL)` (50 ms)
*   `Start` -> `N3 (Batch Load)` (40 ms)
*   `Start` -> `N2 (Streaming Ingestion)` (60 ms)

*   `N1 (ETL)` -> `N5 (NoSQL Write)` (70 ms)
*   `N3 (Batch Load)` -> `N5 (NoSQL Write)` (70 ms)

*   `N1 (ETL)` -> `N4 (Analytics Query)` (100 ms)
*   `N3 (Batch Load)` -> `N4 (Analytics Query)` (100 ms)
*   `N5 (NoSQL Write)` -> `N4 (Analytics Query)` (100 ms)
*   `N2 (Streaming Ingestion)` -> `N4 (Analytics Query)` (100 ms) (for real-time analytics)

*   `N1 (ETL)` -> `N6 (Graph Query)` (90 ms)
*   `N3 (Batch Load)` -> `N6 (Graph Query)` (90 ms)


**Visual Representation of the DAG (Conceptual):**

```mermaid
graph TD
    A[Start] --> B(N1: ETL (50ms))
    A --> C(N3: Batch Load (40ms))
    A --> D(N2: Streaming Ingestion (60ms))

    B --> E(N5: NoSQL Write (70ms))
    C --> E

    B --> F(N4: Analytics Query (100ms))
    C --> F
    E --> F
    D --> F

    B --> G(N6: Graph Query (90ms))
    C --> G

    F --> H[End]
    G --> H
    E --> H
    D --> H
```

**Determining the Critical Path:**
The critical path is the longest path of dependent tasks from the start to the end of the DAG, representing the minimum time required to complete all tasks. We sum the execution times along each possible path.

Let's trace the paths and sum their `Execution Time (ms)`:

1.  **Path 1:** `Start` -> `N1 (50)` -> `N5 (70)` -> `N4 (100)` -> `End`
    *   Total Time: 50 + 70 + 100 = **220 ms**

2.  **Path 2:** `Start` -> `N3 (40)` -> `N5 (70)` -> `N4 (100)` -> `End`
    *   Total Time: 40 + 70 + 100 = **210 ms**

3.  **Path 3:** `Start` -> `N1 (50)` -> `N4 (100)` -> `End`
    *   Total Time: 50 + 100 = **150 ms**

4.  **Path 4:** `Start` -> `N3 (40)` -> `N4 (100)` -> `End`
    *   Total Time: 40 + 100 = **140 ms**

5.  **Path 5:** `Start` -> `N2 (60)` -> `N4 (100)` -> `End`
    *   Total Time: 60 + 100 = **160 ms**

6.  **Path 6:** `Start` -> `N1 (50)` -> `N6 (90)` -> `End`
    *   Total Time: 50 + 90 = **140 ms**

7.  **Path 7:** `Start` -> `N3 (40)` -> `N6 (90)` -> `End`
    *   Total Time: 40 + 90 = **130 ms**


Comparing all path totals, the **critical path** is the one with the maximum execution time:

**Critical Path: `Start` -> `N1 (ETL)` -> `N5 (NoSQL Write)` -> `N4 (Analytics Query)` -> `End`**
**Critical Path Execution Time: 220 ms**

This means that the overall completion time for the entire process, under these assumed dependencies, is dominated by the sequence of ETL, followed by NoSQL Write, and then Analytics Query, taking a total of 220 ms. Any delays in tasks along this path will directly increase the total processing time of the system.

### QUESTION ONE (b): Analyze memory hierarchy and caching effects on throughput, latency, and execution.

**Memory Hierarchy and Caching Effects:**

Memory hierarchy is a tiered system of computer storage that organizes memory based on speed and cost, with faster, smaller, and more expensive memory at the top (e.g., CPU registers, L1/L2/L3 cache) and slower, larger, and cheaper memory at the bottom (e.g., main memory/RAM, disk storage, network storage). Caching is a technique where frequently accessed data is stored in a faster, smaller memory (cache) closer to the CPU to reduce access time to main memory.

Here's how memory hierarchy and caching affect throughput, latency, and execution in the context of the provided Distributed Systems Performance Dataset:

1.  **Throughput (Amount of work done per unit time):**
    *   **Positive Effect:** Effective caching significantly increases throughput. When data is found in a cache (cache hit), the CPU doesn't have to wait for slower memory, allowing it to process instructions and data more quickly. This means more operations per second (higher throughput) can be achieved. For example, if a node like **N4 (Analytics Query)**, which has high `Execution Time` (100ms) and `Memory` (2048MB), frequently accesses the same data for its queries, a good caching strategy will allow it to perform more queries in a given timeframe, leading to higher `Throughput (ops/s)`.
    *   **Negative Effect:** Poor caching or frequent cache misses (data not found in cache) lead to fetching data from slower memory levels, which bottlenecks the CPU and reduces throughput. This is especially true for data-intensive tasks like **N1 (ETL)** or **N3 (Batch Load)** where large datasets might exceed cache capacity, forcing frequent access to slower storage and thus lowering their effective `Throughput`.

2.  **Latency (Time delay between request and response):**
    *   **Positive Effect:** Caching drastically reduces latency. A cache hit means data is available almost immediately, resulting in very low access times. For example, **N2 (Streaming Ingestion)** and **N3 (Batch Load)** have relatively low `Latency` (12ms and 8ms respectively). If their underlying processes can leverage caches effectively, data can be processed and moved with minimal delay, maintaining these low latency figures.
    *   **Negative Effect:** Cache misses increase latency because the system has to retrieve data from lower levels of the memory hierarchy, which could involve main memory, SSD, or even network storage, each adding considerable delay. For tasks like **N4 (Analytics Query)** with a `Latency` of 20ms, or **N6 (Graph Query - Neo4j)** with 18ms, frequent cache misses for graph traversals or analytical computations would significantly inflate these latency figures, making the system less responsive.

3.  **Execution (Overall process of performing a task):**
    *   **Positive Effect:** Caching optimizes execution by ensuring that the CPU is rarely stalled waiting for data. This leads to faster completion of individual tasks and overall applications. Tasks like **N5 (NoSQL Write - Cassandra)**, which has a moderate `Execution Time` (70ms) and `Memory` (1024MB), would benefit from caching frequently accessed data structures or indices, speeding up write operations and reducing the overall `Execution Time`.
    *   **Negative Effect:** Inefficient cache usage (e.g., thrashing, where new data constantly evicts useful cached data) can lead to performance degradation. The CPU spends more time on cache management or waiting, rather than executing core instructions. This can negate the benefits of faster processors and lead to longer `Execution Times` than expected, impacting any of the nodes from **N1** to **N6** if their data access patterns are poorly optimized for caching.

**Summary:**
Optimizing memory hierarchy and leveraging caching mechanisms are crucial for high-performance distributed systems. A well-designed system will utilize multiple levels of caching (CPU cache, OS page cache, application-level caches) to keep frequently used data as close to the processing units as possible. This minimizes memory access latency, maximizes data throughput, and ultimately improves the overall execution speed and efficiency of tasks across the distributed environment, directly impacting the performance metrics (Execution Time, Throughput, Latency) observed in the dataset.

### QUESTION ONE (c): Compare batch processing, streaming systems, and sharding trade-offs.

In distributed systems, choosing the right data processing paradigm and scaling strategy is crucial. This question compares batch processing, streaming systems, and sharding trade-offs, relating them to the provided Distributed Systems Performance Dataset where applicable.

#### 1. Batch Processing

**Description:** Batch processing involves collecting and processing data in large batches over a period. It is typically used for large volumes of historical data where immediate results are not required.

**Characteristics & Trade-offs:**
*   **Latency:** High (e.g., hours to days). Data is processed in bulk, so there's an inherent delay before results are available. `N3 (Batch Load)` in our dataset is characteristic of this, with `Execution Time` of 40ms, preparing data for later use rather than real-time interaction.
*   **Throughput:** High. Optimized for processing large volumes of data efficiently, often leveraging parallel processing. `N3 (Batch Load)` has a high `Throughput` of 1500 ops/s.
*   **Cost Efficiency:** Can be more cost-effective as resources can be provisioned for specific batch windows and then scaled down. Less need for always-on, high-performance infrastructure.
*   **Complexity:** Generally simpler to design and implement compared to streaming, as it deals with finite datasets.
*   **Use Cases:** ETL jobs (`N1 (ETL)`), financial reporting, data warehousing updates, machine learning model training on historical data.
*   **Resilience:** Easier to re-process failed batches.

#### 2. Streaming Systems

**Description:** Streaming systems process data continuously as it arrives, enabling real-time or near real-time analytics and responses. Data is processed record-by-record or in small micro-batches.

**Characteristics & Trade-offs:**
*   **Latency:** Low (e.g., milliseconds to seconds). Designed for immediate processing of incoming data. `N2 (Streaming Ingestion)` in our dataset has a `Latency` of 12ms, reflecting its real-time nature.
*   **Throughput:** Can be high, but managing continuous flow and state can be challenging. `N2 (Streaming Ingestion)` has a `Throughput` of 1000 ops/s.
*   **Cost Efficiency:** Can be more expensive due to the need for always-on infrastructure and potentially higher computational resources for real-time processing.
*   **Complexity:** More complex to design, implement, and maintain due to state management, fault tolerance, and guaranteeing data consistency in real-time.
*   **Use Cases:** Fraud detection, IoT data processing (`N2 (Streaming Ingestion)`), real-time recommendations, monitoring systems, live dashboards.
*   **Resilience:** Requires robust fault-tolerance mechanisms to ensure no data loss and continuous availability.

#### 3. Sharding (Data Partitioning)

**Description:** Sharding is a horizontal partitioning strategy for databases or distributed systems. It involves breaking up a large dataset into smaller, more manageable pieces called shards, which are then distributed across multiple servers or nodes. Each shard can be processed independently.

**Characteristics & Trade-offs:**
*   **Scalability:** Enables horizontal scalability, allowing systems to handle much larger datasets and higher transaction volumes than a single server could. By distributing data and processing across multiple nodes, tasks like `N4 (Analytics Query)` or `N6 (Graph Query)` can execute in parallel, potentially reducing overall execution time.
*   **Performance:** Improves query performance and reduces latency by distributing the workload. A query only needs to access the relevant shard(s) rather than scanning the entire dataset. For `N5 (NoSQL Write - Cassandra)`, sharding is fundamental to its distributed nature, allowing writes to be distributed across many nodes and handled concurrently.
*   **Availability & Fault Tolerance:** If one shard fails, only a portion of the data is affected, and other shards remain operational. Replication across shards (as indicated by `Replicas` in the dataset, e.g., `N1` has 2 `Replicas`, `N2` has 3) further enhances fault tolerance by providing redundant copies of data.
*   **Complexity:** Increases system complexity in terms of data distribution logic, query routing, maintaining consistency across shards, and handling rebalancing or resharding.
*   **Data Consistency:** Ensuring strong consistency across distributed shards can be challenging and may introduce latency trade-offs.
*   **Hotspots:** Poor sharding key selection can lead to data hot spots, where one shard receives disproportionately more traffic, negating the benefits of distribution.
*   **Use Cases:** Large-scale databases (like Cassandra, used by `N5`), distributed file systems, search engines, and any application requiring massive data storage and retrieval capabilities.

**Interplay between Paradigms and Sharding:**
*   Batch processing often benefits from sharding to parallelize the processing of large datasets. Each batch can be split into sub-batches processed on different shards.
*   Streaming systems heavily rely on sharding (often called partitioning in streaming contexts) to distribute incoming data streams and processing logic across a cluster, ensuring high throughput and low latency. For instance, `N2 (Streaming Ingestion)` can partition its incoming stream across multiple nodes for parallel processing.

Understanding these trade-offs is critical for designing efficient and scalable distributed systems, ensuring that the chosen architecture aligns with the application's requirements for speed, consistency, cost, and fault tolerance.

### QUESTION ONE (d): Design fault-tolerant distributed architecture with replication, storage, and scheduling.

Designing a fault-tolerant distributed architecture involves ensuring that the system can continue to operate correctly even if some of its components fail. This requires robust strategies for replication, distributed storage, and intelligent scheduling. Drawing insights from the provided dataset, we can propose a conceptual architecture.

#### Core Principles of Fault Tolerance:
1.  **Redundancy:** Having multiple copies of data or components.
2.  **Isolation:** Limiting the impact of a failure to a single component or subset of components.
3.  **Recovery:** Mechanisms to restore failed components or data.
4.  **Monitoring:** Detecting failures promptly.

#### Proposed Fault-Tolerant Distributed Architecture:

**1. Data Ingestion Layer:**
*   **Components:** This layer handles initial data input, including `N1 (ETL)`, `N2 (Streaming Ingestion)`, and `N3 (Batch Load)`..
*   **Replication:**
    *   **Streaming Ingestion (N2):** Use a distributed messaging queue (e.g., Apache Kafka) with multiple brokers and topic partitions replicated across different nodes. `N2` has `3 Replicas`, which aligns with this for high availability and durability of incoming data streams.
    *   **ETL (N1) & Batch Load (N3):** Input data sources should be replicated (e.g., using distributed file systems or highly available databases). The ETL and Batch processes themselves can be run on fault-tolerant frameworks (e.g., Apache Spark on YARN/Kubernetes) where tasks are retried on failure.
*   **Storage:** Data landed from this layer (intermediate or raw) should be written to a fault-tolerant distributed storage system.

**2. Distributed Storage Layer:**
*   **Components:** This layer provides reliable and scalable storage for all processed and raw data. Examples from the dataset include `N5 (NoSQL Write - Cassandra)` and potentially an underlying distributed file system for other nodes.
*   **Replication:**
    *   **NoSQL Databases (N5 - Cassandra):** Cassandra's inherent architecture provides fault tolerance through data replication across multiple nodes (e.g., `N5` has `3 Replicas`). Data is partitioned (sharded) and replicated according to a replication factor and consistency level. If a node fails, replicas on other nodes ensure data availability.
    *   **Distributed File Systems (e.g., HDFS, S3-compatible storage):** Store data with multiple copies across different data nodes. HDFS, for example, typically replicates data blocks 3 times.
*   **Data Consistency:** Implement eventual consistency for high availability (common in NoSQL) or strong consistency where required, understanding the trade-offs in latency and availability.

**3. Processing & Query Layer:**
*   **Components:** This layer executes data processing, analytics, and serving queries, including `N4 (Analytics Query)` and `N6 (Graph Query - Neo4j)`.
*   **Replication & Redundancy:**
    *   **Compute Nodes:** Run multiple instances of `N4` and `N6` on different physical or virtual machines. `N4` and `N6` both have `2 Replicas`, suggesting active-passive or active-active setups. Load balancers distribute requests and route around failed instances.
    *   **Stateful Services:** If processing tasks maintain state, ensure state is periodically checkpointed to persistent, replicated storage or leverage frameworks that handle state replication (e.g., Flink's checkpointing).
*   **Microservices Architecture:** Decompose services into smaller, independent units. This limits the blast radius of failures (e.g., a failure in `N6` doesn't bring down `N4`).

**4. Distributed Scheduling & Orchestration:**
*   **Components:** A central orchestrator or a distributed consensus mechanism to manage tasks, resource allocation, and detect failures.
*   **Scheduling (`Execution Time` and `Latency`):**
    *   **Dynamic Scheduling:** Use schedulers (e.g., Apache YARN, Kubernetes, Apache Mesos) that can dynamically allocate tasks to healthy nodes, re-schedule failed tasks, and optimize resource utilization. This is critical for managing varying `Execution Times` and `Latency` requirements.
    *   **Resource Isolation:** Ensure tasks are scheduled with sufficient resources (`Memory (MB)`) and isolated to prevent noisy neighbors from affecting critical tasks.
    *   **Priority and Dependencies:** Schedulers should understand task dependencies (as identified in the DAG for Question 1a) and priorities to ensure critical paths are not blocked.
*   **Failure Detection & Recovery:**
    *   **Heartbeating/Health Checks:** Nodes and services periodically report their health status to the orchestrator.
    *   **Automated Restarts:** Failed containers or processes are automatically restarted on healthy nodes.
    *   **Circuit Breakers/Bulkheads:** Prevent cascading failures by isolating failing services and providing fallback mechanisms.
    *   **Leader Election:** For services requiring a single leader (e.g., master nodes), implement leader election protocols (e.g., Apache ZooKeeper, etcd) to automatically elect a new leader upon failure.

**5. Monitoring, Logging, and Alerting:**
*   **Centralized Logging:** Aggregate logs from all distributed components for easier debugging and post-mortem analysis.
*   **Metrics Collection:** Collect performance metrics (`Throughput`, `Latency`, `Execution Time`, `Memory` usage, etc.) from all nodes and services.
*   **Alerting:** Set up alerts for anomalies, threshold breaches, and detected failures to enable quick human intervention if automated recovery fails.

#### Overall Architecture View (Conceptual):

```mermaid
graph TD
    SubGraph Ingestion_Layer
        A[N1: ETL] --> S(Distributed Messaging Queue / DFS)
        B[N2: Streaming Ingestion] --> S
        C[N3: Batch Load] --> S
    End

    S -- Data Flow --> D(Distributed Storage: Cassandra / HDFS)

    SubGraph Processing_Query_Layer
        D -- Data Access --> E[N4: Analytics Query (Replicated)]
        D -- Data Access --> F[N5: NoSQL Write (Replicated)]
        D -- Data Access --> G[N6: Graph Query (Replicated)]
    End

    H[Scheduler/Orchestrator: Kubernetes/YARN] -- Manages/Monitors --> Ingestion_Layer
    H -- Manages/Monitors --> D
    H -- Manages/Monitors --> Processing_Query_Layer

    I[Monitoring & Alerting] -- Feeds from --> Ingestion_Layer
    I -- Feeds from --> D
    I -- Feeds from --> Processing_Query_Layer
    I -- Feeds from --> H
```

This architecture leverages replication at multiple levels (data, compute instances), distributed storage with built-in fault tolerance, and intelligent scheduling to ensure the system can withstand failures and maintain high availability and reliability, as suggested by the `Replicas` column in the provided dataset.


QUESTION TWO

Data Ingestion and Processing Dataset

Task ID	Data Source Type	Dataset Size	Framework	Throughput	Latency	Parallelism
T001	Structured DB	1M	Spark	45000	120	8 GPUs
T002	Streaming IoT	750K	Flink	38500	95	6 GPUs
T003	Graph Data	1.2M	Dask	42000	110	4 GPUs
T004	Logs (Unstructured)	900K	Ray	36000	130	6 GPUs
T005	Sensor Stream	1.1M	Spark	48000	100	8 GPUs
(a)	Evaluate throughput and latency trade-offs across batch and hybrid systems.
(b)	Explain how data sourcing techniques influence data quality and pipeline design.
(c)	Compare scalability of Spark, Flink, Dask, and Ray in heterogeneous environments.
(d)	Apply randomized algorithms for large-scale data summarization and estimation tasks.


### QUESTION TWO (a): Evaluate throughput and latency trade-offs across batch and hybrid systems.

Based on the Data Ingestion and Processing Dataset, we can evaluate the trade-offs between throughput (volume of data processed) and latency (time taken to process a single unit/request):

**1. Batch Systems (e.g., T001, T005 using Spark):**
*   **Performance:** Spark tasks (T001, T005) show the highest throughput in the dataset (45,000 and 48,000 units). However, they also exhibit relatively high latency (120ms and 100ms).
*   **Trade-off:** Batch systems prioritize high-volume processing by grouping data. This leads to efficient hardware utilization (8 GPUs) but introduces a 'wait time' for the batch to form and process, resulting in higher latency compared to pure stream processors.

**2. Hybrid/Streaming Systems (e.g., T002 using Flink, T004 using Ray):**
*   **Performance:** Flink (T002) achieves the lowest latency in the dataset (95ms) with a respectable throughput (38,500). Ray (T004) shows a different profile with high latency (130ms) for unstructured logs, likely due to the complexity of the task.
*   **Trade-off:** Systems like Flink are designed for 'event-at-a-time' processing. This minimizes latency, making them ideal for the 'Sensor Stream' or 'Streaming IoT' use cases where immediate action is required. The trade-off is often a slightly lower overall throughput compared to optimized batch loads because the overhead of processing individual events is higher.

**3. Specialized Processing (e.g., T003 using Dask):**
*   **Performance:** Dask (T003) manages 42,000 throughput with 110ms latency for 1.2M graph data records using only 4 GPUs.
*   **Trade-off:** Dask provides a middle ground, often used for complex data structures like graphs. It demonstrates that with efficient parallelism, one can maintain high throughput for large datasets (1.2M) without the extreme latency penalties seen in unstructured log processing.

**Summary Table of Trade-offs:**

| System Type | Framework | Throughput | Latency | Key Trade-off |
| :--- | :--- | :--- | :--- | :--- |
| **Batch** | Spark | **Highest** | High | Maximum efficiency for large static datasets; poor real-time response. |
| **Streaming** | Flink | Moderate | **Lowest** | Best for real-time IoT/Sensors; higher overhead per record. |
| **Distributed** | Ray/Dask | Moderate | Moderate/High | Flexibility for complex/unstructured tasks; latency varies by compute complexity. |

### QUESTION TWO (b): Explain how data sourcing techniques influence data quality and pipeline design.

Data sourcing is the initial stage of any pipeline. The technique used to acquire data (e.g., Pull vs. Push, Batch vs. Stream) significantly impacts the final quality of the data and the complexity of the architecture.

#### 1. Influence on Data Quality
*   **Structured DB Sourcing (e.g., T001):** Sourcing from traditional databases usually ensures high **Schema Adherence**. Since the source is structured, data types are often pre-validated. However, it can suffer from **Staleness** because the data is only as fresh as the last batch pull.
*   **Streaming IoT & Sensor Streams (e.g., T002, T005):** These provide high **Timeliness** (real-time data). However, they are prone to **Noise** and **Missingness** due to sensor malfunctions or network jitter. Data quality efforts here must focus on deduplication and outlier detection.
*   **Unstructured Logs (e.g., T004):** Sourcing from logs (Ray) often leads to issues with **Consistency**. Log formats may change without notice, requiring robust parsing logic to maintain quality.

#### 2. Influence on Pipeline Design
*   **Ingestion Strategy:**
    *   **Batch Sourcing** (Spark/T001) allows for **Checkpoint-based** pipeline designs where failure recovery involves re-running a specific time-slice.
    *   **Continuous Sourcing** (Flink/T002) requires a **Stateful** design. The pipeline must manage 'watermarks' to handle out-of-order data arrival.
*   **Tool Selection:**
    *   Large, static datasets (1.2M Graph Data) might lead to a design favoring **Dask** for distributed memory management.
    *   High-velocity streams (Sensor Streams) necessitate frameworks like **Spark Streaming** or **Flink** that support back-pressure handling to prevent system crashes during traffic spikes.
*   **Pre-processing Placement:**
    *   For structured sources, cleaning often happens *after* loading (ELT).
    *   For noisy IoT sources, cleaning/filtering usually happens *during* ingestion (ETL) to avoid storing 'garbage' data in the primary storage layer.

**Summary:**
The choice of sourcing technique dictates whether the pipeline design must prioritize **Throughput** (Batch) or **Low Latency** (Streaming), and whether the quality control mechanisms need to be **Reactive** (fixing data in the warehouse) or **Proactive** (filtering data at the edge).

### QUESTION TWO (c): Compare scalability of Spark, Flink, Dask, and Ray in heterogeneous environments.

Heterogeneous environments involve clusters with varying hardware (different CPU/GPU counts, memory sizes, or network speeds). Scalability in these environments depends on how frameworks handle resource allocation and task scheduling.

#### 1. Apache Spark
*   **Scalability:** Highly scalable for massive batch processing and micro-batch streaming. It excels at horizontal scaling (adding more nodes).
*   **Heterogeneous Performance:** Historically rigid; it assumes uniform executor sizes. However, with modern resource managers like Kubernetes or YARN, it can handle heterogeneous clusters by defining specific node selectors, though task skew can still occur if one node is significantly slower.

#### 2. Apache Flink
*   **Scalability:** Exceptional for stateful stream processing. It scales linearly by partitioning data streams.
*   **Heterogeneous Performance:** Flink's slots-based resource model allows for some flexibility. In heterogeneous setups, Flink's ability to handle back-pressure is a major advantage, ensuring that slower nodes (e.g., those with fewer GPUs as seen in T002) don't crash the entire pipeline, though they may become the bottleneck for the overall throughput.

#### 3. Dask
*   **Scalability:** Excellent for Python-native scientific computing and shared-memory tasks. It scales from a single laptop to thousands of nodes.
*   **Heterogeneous Performance:** Dask is very flexible with heterogeneous hardware. Its scheduler is sophisticated enough to track resource tags (like `GPU` or `Memory`). As seen in **T003**, Dask can achieve high throughput (42,000) even with fewer GPUs (4) by efficiently managing the task graph based on available resources.

#### 4. Ray
*   **Scalability:** Designed specifically for distributed AI and reinforcement learning. It has very low-latency task scheduling.
*   **Heterogeneous Performance:** Ray is perhaps the most robust for heterogeneous environments. It uses a "Resource-Aware" scheduler where tasks can explicitly request specific amounts of CPUs, GPUs, or custom resources. This makes it ideal for **T004**, where unstructured logs might require heavy CPU parsing before GPU-bound processing.

| Framework | Scaling Strength | Heterogeneous Fit | Best Use Case |
| :--- | :--- | :--- | :--- |
| **Spark** | Data Volume (Batch) | Moderate | Large-scale ETL/Data Warehousing |
| **Flink** | Low-Latency State | Moderate | Real-time Analytics/IoT |
| **Dask** | Complex Math/Python | High | Parallel Data Science/Graph Data |
| **Ray** | Task Granularity/AI | **Highest** | Distributed Training/Microservices |

### QUESTION TWO (d): Apply randomized algorithms for large-scale data summarization and estimation tasks.

When dealing with datasets at the scale of 1M+ records (as seen in T001-T005), processing every single item exactly can be computationally expensive. Randomized algorithms provide approximate answers with high probability, significantly reducing memory and time requirements.

#### 1. Cardinality Estimation (HyperLogLog)
*   **Application:** In **T001 (Structured DB)** or **T002 (Streaming IoT)**, we often need to count unique visitors or unique device IDs.
*   **How it works:** Instead of storing every unique ID in a hash set (which grows linearly with data), HyperLogLog uses hashing and bit-pattern analysis to estimate the count of distinct elements with a very small memory footprint (typically a few KB for millions of items).

#### 2. Frequency Estimation (Count-Min Sketch)
*   **Application:** Useful for **T004 (Logs)** to find the most frequent error messages or 'heavy hitters' without storing every log entry.
*   **How it works:** It uses a 2D array and multiple hash functions to map elements. While it can have collisions (causing slight overestimation), it provides a fixed-size summary of item frequencies regardless of the number of unique items encountered.

#### 3. Membership Filtering (Bloom Filters)
*   **Application:** Used in **T005 (Sensor Stream)** to quickly check if a specific sensor reading has been seen before or to filter out known malicious IPs in log processing.
*   **How it works:** A space-efficient probabilistic data structure that tells you if an element is *definitely not* in a set or *possibly* in a set. It never has false negatives.

#### 4. Reservoir Sampling
*   **Application:** Essential for **T002 (Streaming IoT)** where the data is an infinite stream.
*   **How it works:** It allows for maintaining a representative random sample of size $k$ from a stream of unknown length, ensuring every item has an equal probability of being in the final sample. This sample can then be used for offline statistical analysis.

**Trade-offs of Randomized Algorithms:**
*   **Pros:** Significant reduction in memory (sub-linear), constant time complexity $O(1)$ for updates, and high horizontal scalability.
*   **Cons:** Results are approximate (not exact) and they introduce a 'probability of error' that must be tuned based on the application's tolerance.


QUESTION THREE

Machine Learning Systems Dataset


Project ID	Model	Dataset Size	Accuracy	Training Time	Storage System
P001	XGBoost	1.2M	0.91	12 hrs	MongoDB
P002	LightGBM	800K	0.88	10 hrs	Cassandra
P003	CatBoost	950K	0.89	11 hrs	Neo4j
P004	NeuralNet	1.5M	0.93	18 hrs	Distributed FS
P005	PINNs	600K	0.87	15 hrs	MongoDB
(a)	Evaluate dataset size and storage model impact on model performance.
(b)	Compare MongoDB, Cassandra, and Neo4j for scalability and query efficiency.
(c)	Assess differential privacy and federated learning impact on accuracy and computation.
(d)	Analyze training time implications for model selection in large-scale systems.

### QUESTION THREE (a): Evaluate dataset size and storage model impact on model performance.

Based on the Machine Learning Systems Dataset, we can observe the following impacts:

**1. Dataset Size and Accuracy:**
*   There is a general positive correlation between dataset size and model accuracy.
*   **P004 (NeuralNet)** uses the largest dataset (1.5M) and achieves the highest accuracy (0.93). This reflects the ability of deep learning models to effectively leverage large-scale data.
*   **P005 (PINNs)**, despite having the smallest dataset (600K), has a high training time (15 hrs), indicating that model complexity also plays a significant role in performance and resource consumption, not just data volume.

**2. Storage Model Impact:**
*   **Distributed FS (P004):** The use of a Distributed File System for the largest dataset (NeuralNet) suggests that for high-volume, high-throughput training, flat-file distributed storage is preferred over structured NoSQL databases to avoid database overhead during training epochs.
*   **NoSQL (MongoDB/Cassandra - P001, P002):** These are used for structured gradient boosting models (XGBoost, LightGBM). These storage systems provide efficient access to feature vectors, supporting high accuracy (0.91 and 0.88) while maintaining manageable training times.
*   **Graph DB (Neo4j - P003):** Used for CatBoost (950K). This suggests the model might be leveraging relational or graph-based features. The storage system choice here is likely driven by the need to efficiently query complex relationships rather than raw training throughput.

**Summary:** Larger datasets generally lead to better accuracy, but the storage system must be matched to the data structure and access patterns of the model to maintain training efficiency.

### QUESTION THREE (d): Analyze training time implications for model selection in large-scale systems.

In large-scale distributed systems, training time is a critical factor for model selection, often acting as a constraint alongside accuracy. Analyzing the dataset reveals several implications:

**1. Diminishing Returns on Accuracy vs. Time:**
*   **P004 (NeuralNet)** achieves the highest accuracy (**0.93**) but requires **18 hours**.
*   In contrast, **P001 (XGBoost)** achieves **0.91** accuracy in only **12 hours**.
*   **Implication:** For many industrial applications, a 2% accuracy gain might not justify a 50% increase in training time and resource consumption. Systems requiring frequent model retraining (e.g., daily updates) might favor XGBoost or LightGBM for their better 'accuracy-per-hour' ratio.

**2. Iteration Speed and Development Cycles:**
*   Lower training times (like **P002's LightGBM at 10 hrs**) allow for faster hyperparameter tuning and experimentation. In a production environment, being able to iterate twice a day vs. once a day (like NeuralNet) can lead to a more robust model over time through more exhaustive optimization.

**3. Resource Cost and Scaling:**
*   Training time directly correlates with cloud compute costs. Long-running tasks like **P005 (PINNs at 15 hrs for a small dataset)** suggest high computational complexity (likely due to solving differential equations).
*   **Implication:** If a model is both slow to train and hard to parallelize, it becomes a bottleneck. Models that support distributed training (like XGBoost or NeuralNets on Distributed FS) are preferred even if their raw training time is high, provided they scale linearly with more hardware.

**4. Cold Start and Recovery Time:**
*   In the event of a system failure, the 'Recovery Time Objective' (RTO) for an ML pipeline depends on training time. A model that takes 18 hours to retrain from scratch poses a higher risk to business continuity than one that takes 10 hours.

**Conclusion:** Model selection is not just about picking the highest accuracy. It is a multi-objective optimization problem involving **Accuracy**, **Training Latency**, and **Compute Budget**. In the provided dataset, **LightGBM (P002)** stands out as a highly efficient choice for balance, while **NeuralNet (P004)** is the choice when performance is the absolute priority regardless of temporal cost.

### QUESTION THREE (c): Assess differential privacy and federated learning impact on accuracy and computation.

In large-scale machine learning systems, privacy-preserving techniques are often required to protect sensitive data. Here is an assessment of their impact:

#### 1. Differential Privacy (DP)
*   **Impact on Accuracy:** DP introduces 'noise' into the training process (e.g., during gradient updates). This creates a **privacy-utility trade-off**: stronger privacy guarantees (lower epsilon) typically lead to lower model accuracy because the model cannot capture fine-grained patterns in the data as easily.
*   **Impact on Computation:** Implementing DP requires extra steps like gradient clipping and noise addition. This increases the computational cost per training step, although it doesn't necessarily change the overall complexity class of the algorithm.

#### 2. Federated Learning (FL)
*   **Impact on Accuracy:** FL trains models across decentralized devices (e.g., mobile phones) without moving raw data. Accuracy can be impacted by **Non-IID data distribution**, where data on different devices is not representative of the overall population, making model convergence harder compared to centralized training (like P004's NeuralNet).
*   **Impact on Computation:** FL significantly changes the computational profile. Instead of high-speed intra-cluster communication, the bottleneck becomes **network communication** between the central server and remote devices. It requires sophisticated aggregation algorithms (like FedAvg) and can be slow if devices have limited bandwidth or battery.

**Integration into the Dataset Context:**
If **P004 (NeuralNet)** were to implement Federated Learning with Differential Privacy, we would likely see the **Accuracy (0.93)** decrease due to noise, and the **Training Time (18 hrs)** increase significantly due to the communication overhead across distributed nodes.

### QUESTION THREE (b): Compare MongoDB, Cassandra, and Neo4j for scalability and query efficiency.

In the context of the ML systems dataset, different storage backends were chosen for different model types. Here is a comparison of their performance characteristics:

#### 1. MongoDB (Used in P001, P005)
*   **Type:** Document-oriented NoSQL database.
*   **Scalability:** Scales horizontally through **sharding**. It is excellent for handling semi-structured data where the schema might evolve (e.g., storing varied metadata for XGBoost or PINNs).
*   **Query Efficiency:** Highly efficient for CRUD operations and range queries on indexed fields. However, complex joins are less efficient than in relational or graph databases. It is ideal for fetching feature vectors by ID.

#### 2. Cassandra (Used in P002)
*   **Type:** Wide-column store based on the Dynamo/BigTable models.
*   **Scalability:** Exceptional horizontal scalability with a **masterless architecture**. It is designed for high-write availability across multiple data centers. This makes it suitable for P002 (LightGBM) if the training data is being ingested at a very high velocity.
*   **Query Efficiency:** Extremely fast for writes and simple lookups by partition key. It is less flexible for complex ad-hoc queries compared to MongoDB, as the query pattern must be defined by the data model (schema-on-write).

#### 3. Neo4j (Used in P003)
*   **Type:** Native Graph Database.
*   **Scalability:** Historically stronger at vertical scaling, but modern versions support causal clustering for horizontal read scalability. It is more specialized than MongoDB or Cassandra.
*   **Query Efficiency:** Unmatched efficiency for **relational traversal** (e.g., finding connections between nodes). For P003 (CatBoost), if the features involve 'who is connected to whom' or 'pathway analysis,' Neo4j allows the model to retrieve these complex relational features much faster than a JOIN-heavy SQL or NoSQL query.

**Summary Comparison Table:**

| Feature | MongoDB | Cassandra | Neo4j |
| :--- | :--- | :--- | :--- |
| **Data Model** | JSON-like Documents | Wide-column / Tabular | Nodes and Edges (Graph) |
| **Best Scalability** | Horizontal (Sharding) | **Highest** (Peer-to-peer) | Vertical / Clustering |
| **Query Strength** | Flexible ad-hoc queries | High-velocity writes | **Relationship traversal** |
| **ML Use Case** | General Metadata/Features | High-scale time-series/logs | Relational/Social features |